# Import Dependencies

In [ ]:
import os
import torch
import open_clip
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch.optim as optim
import torch.nn as nn
from open_clip import create_model_from_pretrained, get_tokenizer


# Load BiomedCLIP

In [ ]:

MODEL_NAME = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'


# Load the model and config files from the Hugging Face Hub
model, preprocess = create_model_from_pretrained(MODEL_NAME)
tokenizer = get_tokenizer(MODEL_NAME)

# Load the Processor and Model
# processor = AutoProcessor.from_pretrained('microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')

# device = torch.device("cpu")
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f'Using device: {device}')
model.to(device)
# model.head  = nn.Sequential(
    # nn.Linear(768, 512),
# )
# model.eval()


In [ ]:
# Define Transformations
transform = transforms.Compose([
    # transforms.RandAugment(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.481, 0.457, 0.408], std=[0.268, 0.261, 0.275])
])

In [ ]:
import numpy as np
DATASET_PATH="../data/DIBaS_Dataset"

# Custom Dataset Class
class DIBaSDataset(Dataset):
    def __init__(self, dataset_path, transform=None):
        self.dataset_path = dataset_path
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.classes = sorted(os.listdir(dataset_path))  # Sorted for consistency
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}

        # Load all image paths and labels
        for class_name in self.classes:
            class_dir = os.path.join(dataset_path, class_name)
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir):
                    img_path = os.path.join(class_dir, img_name)
                    self.image_paths.append(img_path)
                    self.labels.append(class_name)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        # Load and transform image
        image = Image.open(img_path).convert("RGB")
        #print(np.array(Image.open(img_path)).shape)
        if self.transform:
            image = self.transform(image)

        # Convert text label to tokenized form
        text = torch.squeeze(tokenizer([label]))  # Tokenize class name

        return image, text, self.class_to_idx[label]

# Load Dataset
dataset = DIBaSDataset(DATASET_PATH, transform=transform)
print(len(dataset))
dataloader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=4)


In [ ]:
classifier = nn.Sequential(
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Linear(512, len(dataset.class_to_idx)),
)
classifier.to(device)


In [ ]:
# Define Optimizer & Loss Function
# optimizer = optim.AdamW([*model.visual.head.parameters(), *model.text.proj.parameters()], lr=5e-5, weight_decay=0.01)
optimizer = optim.Adam(classifier.parameters(), lr=3e-3, weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
# Fine-Tuning Loop
num_epochs = 4
#model.head.train()
#model.train()

In [ ]:
for epoch in range(num_epochs):
    total_loss = 0.0
    for images, texts, labels in dataloader:
        images, labels = images.to(device), torch.tensor(labels).to(device)
        texts = texts.to(device)

        # Get Image & Text Features
        image_features = model.encode_image(images)
        text_features = model.encode_text(texts)

        # print(labels)
        # Compute Similarity Scores
        # logits_per_image = (image_features @ text_features.T)  # Cosine similarity
        inputs = torch.cat((image_features, text_features), 1)
        logits_per_image = classifier(inputs) 
        loss = loss_fn(logits_per_image, labels)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss:.4f}")

# Save the fine-tuned model
torch.save(classifier.state_dict(), "../models/openclip_dibas_finetuned.pth")
print("Model saved successfully!")
